In [1]:
# instalar statsforecast
# !pip install statsforecast

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Comparacao do Metodo Theta com metodos estatisticos tradicionais para
previsao de niveis de rio (dados no formato "Cotas" da ANA/Hidroweb).

O QUE ESTE SCRIPT FAZ
----------------------
1. Le o CSV bruto exportado do Hidroweb/ANA (formato "Cotas", separador ';')
   e reconstroi uma serie diaria de nivel do rio (cm).
2. Agrega para uma serie MENSAL (media), que e a frequencia recomendada para
   comparar Theta com ARIMA/SARIMA e ETS: mantem o ciclo sazonal anual (cheia/
   seca) com uma serie de tamanho tratavel (o Theta, assim como os demais
   metodos aqui, foi originalmente avaliado sobre series mensais/trimestrais/
   anuais na competicao M3/M4).
3. Faz uma validacao cruzada do tipo "rolling origin" (walk-forward, janela
   expansiva) com varias origens de previsao, nao apenas um unico split
   treino/teste — isso evita conclusoes tiradas de um unico recorte "sortudo"
   e e a pratica padrao em avaliacao de metodos de previsao.
4. Ajusta, em cada origem, os seguintes metodos:
      Familia Theta:
      - Theta classico (statsmodels ThetaModel; Assimakopoulos & Nikolopoulos, 2000)
      - Theta Otimizado / OTM (statsforecast; Fiorucci et al., 2016) — em vez de
        fixar theta=2 e os pesos das duas "theta lines" em 0.5/0.5 como no
        classico, o OTM OTIMIZA o parametro theta, os pesos da combinacao e o
        alpha do SES por minimos quadrados sobre o proprio treino.
      - Dynamic Optimized Theta Model / DOTM (statsforecast; Fiorucci et al.,
        2016) — reformula o Theta como um modelo dinamico (espaco de estados)
        em vez da formula estatica SES+deriva, permitindo que a contribuicao
        da tendencia se ajuste ao longo do tempo; tambem otimiza os parametros.
      Metodos tradicionais (benchmark):
      - SARIMA (statsmodels SARIMAX, ordem escolhida uma vez via grid + AIC)
      - ETS / Holt-Winters (statsmodels ExponentialSmoothing)
      - Naive (repete o ultimo valor observado)
      - Naive Sazonal (repete o valor do mesmo mes do ciclo anterior)
5. Calcula métricas de erro (MAE, RMSE, MASE) agregadas por metodo e por
   horizonte de previsao.
6. Aplica o teste de Diebold-Mariano (cada variante Theta vs. cada metodo
   tradicional) para verificar se a diferenca de desempenho e estatisticamente
   significativa, e nao apenas fruto de acaso.
7. Gera graficos comparativos e tabelas-resumo, salvos em disco.

IMPORTANTE — HONESTIDADE ESTATISTICA
-------------------------------------
Este script NAO forca nenhuma variante do Theta a "vencer". Ele faz uma
comparacao justa e reproduzivel; a familia Theta e reconhecidamente muito
competitiva (o Theta classico venceu a competicao M3; OTM e DOTM tendem a
superar o classico em varios estudos), mas o resultado real depende dos
dados da sua estacao. Rode o script e deixe os numeros falarem — se algum
metodo tradicional vencer, isso aparecera nas tabelas e nos graficos, com o
devido teste de significancia estatistica.

DEPENDENCIAS
-------------
pip install --break-system-packages pandas numpy scipy statsmodels matplotlib statsforecast

USO
----
Ajuste as constantes na secao CONFIG e rode:
    python3 theta_vs_metodos_tradicionais.py
"""

from __future__ import annotations

import calendar
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # nao depende de display grafico
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsforecast.models import OptimizedTheta, DynamicOptimizedTheta

warnings.filterwarnings("ignore")  # statsmodels e verboso com avisos de convergencia

# ======================================================================
# CONFIG — ajuste aqui para o seu caso
# ======================================================================

CSV_PATH = "10100000_Cotas.csv"          # caminho do CSV exportado do Hidroweb/ANA
ESTACAO_CODIGO = None                     # None = usa a unica estacao do arquivo;
                                           # ou informe o codigo (str) se houver mais de uma
COLUNA_VALOR = "nivel_cm"                 # nome dado a serie de nivel (cm)

FREQ = "MS"                               # 'MS' = mensal (recomendado); 'D' = diario
PERIODO_SAZONAL = 12                      # 12 para mensal; 7 (semana) ou 365 para diario

HORIZONTE = 12                            # passos a frente previstos em cada origem
N_ORIGENS = 6                             # numero de origens na validacao rolling-origin
PASSO_ENTRE_ORIGENS = 6                   # espacamento (em periodos) entre origens sucessivas
MIN_TREINO = 120                          # minimo de observacoes no treino da 1a origem

OUTPUT_DIR = Path("resultados_theta2")

# Paleta de cores fixa (uma cor por serie, nunca reaproveitada) — acessivel
# para daltonismo, seguindo ordem categorica fixa.
COR_REAL = "#0b0b0b"
CORES_MODELOS = {
    "Theta": "#2a78d6",                    # azul
    "Theta Otimizado": "#4a3aa7",           # violeta
    "DOTM": "#e34948",                      # vermelho
    "SARIMA": "#eb6834",                    # laranja
    "ETS (Holt-Winters)": "#1baf7a",        # verde-agua
    "Naive": "#eda100",                     # amarelo
    "Naive Sazonal": "#e87ba4",             # magenta
}
COR_GRID = "#e1e0d9"
COR_EIXO = "#898781"

# Agrupamento usado nos graficos/testes (familia Theta vs. benchmarks
# tradicionais) — mantendo a ordem de exibicao acima.
MODELOS_THETA = ["Theta", "Theta Otimizado", "DOTM"]
MODELOS_TRADICIONAIS = ["SARIMA", "ETS (Holt-Winters)", "Naive", "Naive Sazonal"]




In [2]:
# ======================================================================
# 1) LEITURA E PARSING DO CSV BRUTO (formato "Cotas" da ANA/Hidroweb)
# ======================================================================

def carregar_serie_diaria_ana(caminho_csv: str, estacao_codigo: str | None = None) -> pd.Series:
    """
    Le um export de "Cotas" do Hidroweb/ANA e devolve uma serie diaria de
    nivel do rio (cm), indexada por data.

    Estrutura do arquivo (separador ';'):
      - Cada linha e um MES (coluna 'Data' = 01/MM/AAAA).
      - Linhas com 'hora' preenchida (07:00, 17:00, ...) sao LEITURAS
        pontuais e sao ignoradas aqui.
      - A linha do mes com 'hora' em branco e 'MediaDiaria' == 1 traz a
        MEDIA DIARIA do nivel em 31 colunas: Cota01 ... Cota31 (uma por dia
        do mes; dias inexistentes ficam vazios).
      - 'NivelConsistencia' pode ser 1 (dado bruto) ou 2 (dado consistido/
        validado); quando os dois existem para o mesmo mes, o consistido
        (2) e preferido.
    """
    df = pd.read_csv(caminho_csv, sep=";", encoding="latin1", dtype=str)

    if estacao_codigo is not None:
        df = df[df["EstacaoCodigo"] == str(estacao_codigo)]
    if df.empty:
        raise ValueError("Nenhuma linha encontrada para a estacao informada.")

    # mantem apenas a linha "resumo mensal" (hora em branco -> MediaDiaria==1)
    hora = df["hora"].fillna("").str.strip()
    mensal = df[hora == ""].copy()

    mensal["NivelConsistencia"] = mensal["NivelConsistencia"].astype(int)
    mensal["dt"] = pd.to_datetime(mensal["Data"], format="%d/%m/%Y")

    # se ha duas linhas para o mesmo mes (consistencia 1 e 2), fica com a 2
    mensal = mensal.sort_values(["dt", "NivelConsistencia"], ascending=[True, False])
    mensal = mensal.drop_duplicates(subset="dt", keep="first")

    cota_cols = [f"Cota{str(d).zfill(2)}" for d in range(1, 32)]

    registros = {}
    for _, linha in mensal.iterrows():
        ano, mes = linha["dt"].year, linha["dt"].month
        n_dias = calendar.monthrange(ano, mes)[1]
        for dia in range(1, n_dias + 1):
            valor = linha[f"Cota{str(dia).zfill(2)}"]
            if pd.isna(valor) or valor == "":
                continue
            registros[pd.Timestamp(year=ano, month=mes, day=dia)] = float(valor)

    if not registros:
        raise ValueError("Nao foi possivel extrair nenhum valor diario do CSV.")

    serie = pd.Series(registros).sort_index()
    serie.index.name = "data"
    serie.name = COLUNA_VALOR
    return serie


def preparar_serie(serie_diaria: pd.Series, freq: str = "MS") -> pd.Series:
    """
    Agrega a serie diaria para a frequencia desejada e trata pequenas
    lacunas por interpolacao temporal (no maximo 2 periodos seguidos
    ausentes; lacunas maiores sao descartadas para nao inventar dado).
    """
    if freq == "D":
        s = serie_diaria.asfreq("D")
    else:
        s = serie_diaria.resample(freq).mean()
        s = s.asfreq(freq)

    n_antes = s.isna().sum()
    s = s.interpolate(method="time", limit=2)
    n_depois = s.isna().sum()
    print(f"[preparar_serie] lacunas antes={n_antes}, depois da interpolacao={n_depois}")

    s = s.dropna()
    return s




In [3]:
# ======================================================================
# 2) MODELOS DE PREVISAO — cada funcao recebe a serie de treino e o
#    horizonte h, e devolve um np.ndarray com h previsoes.
# ======================================================================

def prever_theta(treino: pd.Series, h: int, periodo: int) -> np.ndarray:
    """Metodo Theta classico (Assimakopoulos & Nikolopoulos, 2000), via
    statsmodels. method='auto' escolhe decomposicao aditiva ou
    multiplicativa automaticamente (importante aqui, pois o nivel do rio
    pode assumir valores negativos, o que inviabiliza a multiplicativa)."""
    modelo = ThetaModel(treino, period=periodo, method="auto")
    resultado = modelo.fit()
    return resultado.forecast(h).to_numpy()


def prever_theta_otimizado(treino: pd.Series, h: int, periodo: int) -> np.ndarray:
    """Optimized Theta Method — OTM (Fiorucci, Pellegrini, Louzada, Petropoulos
    & Koehler, 2016, 'Models for optimising the theta method and their
    relationship to state space models', IJF). Implementacao: statsforecast
    (Nixtla), a biblioteca de referencia em Python para essa familia de
    metodos. Ao inves de fixar theta=2 e combinar as duas 'theta lines' com
    pesos 0.5/0.5 (como no classico), o OTM otimiza theta, os pesos da
    combinacao e o alpha do SES minimizando o erro quadratico no treino —
    e por isso tende a se adaptar melhor a cada serie especifica.
    Usamos decomposicao aditiva (nao multiplicativa) porque o nivel do rio
    pode ser negativo."""
    y = treino.to_numpy(dtype=float)
    modelo = OptimizedTheta(season_length=periodo, decomposition_type="additive")
    modelo = modelo.fit(y)
    return np.asarray(modelo.predict(h=h)["mean"])


def prever_dotm(treino: pd.Series, h: int, periodo: int) -> np.ndarray:
    """Dynamic Optimized Theta Model — DOTM (Fiorucci et al., 2016, mesma
    referencia do OTM). Implementacao: statsforecast (Nixtla). Reformula o
    Theta como um modelo dinamico/espaco de estados (em vez da formula
    fechada SES + deriva linear), permitindo que a contribuicao da tendencia
    varie ao longo da serie, alem de otimizar os parametros como o OTM. E
    geralmente apontado como a variante mais precisa da familia Theta nos
    estudos de Fiorucci & Louzada. Decomposicao aditiva pelo mesmo motivo
    do OTM (nivel do rio pode ser negativo)."""
    y = treino.to_numpy(dtype=float)
    modelo = DynamicOptimizedTheta(season_length=periodo, decomposition_type="additive")
    modelo = modelo.fit(y)
    return np.asarray(modelo.predict(h=h)["mean"])


def _escolher_ordem_sarima(treino: pd.Series, periodo: int):
    """Faz uma pequena busca em grade (poucas combinacoes) e escolhe a
    ordem SARIMA com menor AIC no conjunto de treino inicial. E feito
    UMA VEZ (nao a cada origem da validacao) para manter o custo
    computacional razoavel — pratica comum ao 'congelar' a ordem do
    modelo e apenas reajustar os coeficientes a cada janela."""
    melhor_aic = np.inf
    melhor_ordem = ((1, 1, 1), (0, 1, 1, periodo))
    candidatos_ordem = [(0, 1, 1), (1, 1, 0), (1, 1, 1), (2, 1, 1)]
    candidatos_sazonais = [(0, 1, 1, periodo), (1, 1, 0, periodo), (1, 1, 1, periodo)]
    for ordem in candidatos_ordem:
        for ordem_saz in candidatos_sazonais:
            try:
                modelo = SARIMAX(
                    treino, order=ordem, seasonal_order=ordem_saz,
                    enforce_stationarity=False, enforce_invertibility=False,
                )
                res = modelo.fit(disp=False)
                if res.aic < melhor_aic:
                    melhor_aic = res.aic
                    melhor_ordem = (ordem, ordem_saz)
            except Exception:
                continue
    print(f"[SARIMA] ordem escolhida por AIC: {melhor_ordem} (AIC={melhor_aic:.1f})")
    return melhor_ordem


def prever_sarima(treino: pd.Series, h: int, ordem, ordem_sazonal) -> np.ndarray:
    modelo = SARIMAX(
        treino, order=ordem, seasonal_order=ordem_sazonal,
        enforce_stationarity=False, enforce_invertibility=False,
    )
    res = modelo.fit(disp=False)
    return res.get_forecast(h).predicted_mean.to_numpy()


def prever_ets(treino: pd.Series, h: int, periodo: int) -> np.ndarray:
    """ETS / Holt-Winters aditivo com tendencia amortecida. Usa componentes
    ADITIVOS (nao multiplicativos) porque o nivel do rio pode ser negativo."""
    modelo = ExponentialSmoothing(
        treino, trend="add", damped_trend=True,
        seasonal="add", seasonal_periods=periodo,
        initialization_method="estimated",
    )
    res = modelo.fit()
    return res.forecast(h).to_numpy()


def prever_naive(treino: pd.Series, h: int) -> np.ndarray:
    return np.repeat(treino.iloc[-1], h)


def prever_naive_sazonal(treino: pd.Series, h: int, periodo: int) -> np.ndarray:
    ultimos = treino.iloc[-periodo:].to_numpy()
    reps = int(np.ceil(h / periodo))
    return np.tile(ultimos, reps)[:h]




In [4]:
# ======================================================================
# 3) VALIDACAO ROLLING-ORIGIN (walk-forward, janela expansiva)
# ======================================================================

@dataclass
class ResultadoFold:
    origem: pd.Timestamp
    modelo: str
    passo_h: np.ndarray       # 1..h
    data_alvo: pd.DatetimeIndex
    y_real: np.ndarray
    y_previsto: np.ndarray


def validar_rolling_origin(
    serie: pd.Series,
    h: int,
    n_origens: int,
    passo: int,
    min_treino: int,
    periodo: int,
) -> list[ResultadoFold]:
    n = len(serie)
    ultima_origem_idx = n - h
    primeira_origem_idx = ultima_origem_idx - (n_origens - 1) * passo
    if primeira_origem_idx < min_treino:
        raise ValueError(
            f"Serie curta demais para {n_origens} origens com h={h} e "
            f"min_treino={min_treino}. Reduza N_ORIGENS, HORIZONTE ou "
            f"PASSO_ENTRE_ORIGENS."
        )

    origens_idx = list(range(primeira_origem_idx, ultima_origem_idx + 1, passo))
    print(f"[rolling-origin] {len(origens_idx)} origens, treino inicial={origens_idx[0]} obs, "
          f"treino final={origens_idx[-1]} obs, horizonte={h}")

    # ordem do SARIMA escolhida uma unica vez, com o treino da 1a origem
    ordem, ordem_sazonal = _escolher_ordem_sarima(serie.iloc[:origens_idx[0]], periodo)

    resultados: list[ResultadoFold] = []
    for origem_idx in origens_idx:
        treino = serie.iloc[:origem_idx]
        teste = serie.iloc[origem_idx: origem_idx + h]
        origem_data = treino.index[-1]

        previsoes = {
            "Theta": prever_theta(treino, h, periodo),
            "Theta Otimizado": prever_theta_otimizado(treino, h, periodo),
            "DOTM": prever_dotm(treino, h, periodo),
            "SARIMA": prever_sarima(treino, h, ordem, ordem_sazonal),
            "ETS (Holt-Winters)": prever_ets(treino, h, periodo),
            "Naive": prever_naive(treino, h),
            "Naive Sazonal": prever_naive_sazonal(treino, h, periodo),
        }

        for nome_modelo, y_hat in previsoes.items():
            resultados.append(ResultadoFold(
                origem=origem_data,
                modelo=nome_modelo,
                passo_h=np.arange(1, h + 1),
                data_alvo=teste.index,
                y_real=teste.to_numpy(),
                y_previsto=np.asarray(y_hat),
            ))
        print(f"  origem {origem_data.date()} (treino={len(treino)} obs) — OK")

    return resultados




In [5]:
# ======================================================================
# 4) METRICAS DE ERRO
# ======================================================================

def mae(y, yhat):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))


def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))


def mase(y, yhat, y_treino_hist: np.ndarray, periodo: int):
    """Mean Absolute Scaled Error (Hyndman & Koehler, 2006).
    Preferido a MAPE aqui porque o nivel do rio cruza o zero (MAPE nao e
    definido/fica instavel quando o valor real esta perto de 0)."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    diffs_sazonais = np.abs(y_treino_hist[periodo:] - y_treino_hist[:-periodo])
    escala = np.mean(diffs_sazonais)
    if escala == 0 or np.isnan(escala):
        return np.nan
    return float(np.mean(np.abs(y - yhat)) / escala)


def montar_tabela_metricas(resultados: list[ResultadoFold], serie: pd.Series, periodo: int) -> pd.DataFrame:
    linhas = []
    for r in resultados:
        hist_treino = serie.loc[: r.origem].to_numpy()
        linhas.append({
            "modelo": r.modelo,
            "origem": r.origem,
            "MAE": mae(r.y_real, r.y_previsto),
            "RMSE": rmse(r.y_real, r.y_previsto),
            "MASE": mase(r.y_real, r.y_previsto, hist_treino, periodo),
        })
    df = pd.DataFrame(linhas)
    resumo = df.groupby("modelo")[["MAE", "RMSE", "MASE"]].mean().sort_values("MASE")
    return df, resumo


def montar_tabela_por_horizonte(resultados: list[ResultadoFold]) -> pd.DataFrame:
    linhas = []
    for r in resultados:
        for passo, real, prev in zip(r.passo_h, r.y_real, r.y_previsto):
            linhas.append({
                "modelo": r.modelo, "origem": r.origem, "h": passo,
                "erro_abs": abs(real - prev),
            })
    df = pd.DataFrame(linhas)
    return df.groupby(["modelo", "h"])["erro_abs"].mean().reset_index()




In [6]:
# ======================================================================
# 5) TESTE DE DIEBOLD-MARIANO (significancia estatistica)
# ======================================================================

def teste_diebold_mariano(erro1: np.ndarray, erro2: np.ndarray, h: int, loss: str = "abs"):
    """
    Testa se o erro medio do modelo 1 e estatisticamente diferente (menor)
    do erro do modelo 2, usando o teste de Diebold & Mariano (1995) com a
    correcao de amostra pequena de Harvey, Leybourne & Newbold (1997) e
    variancia HAC (Newey-West) com defasagem h-1 — necessaria porque
    previsoes de h passos se sobrepoem e os erros ficam autocorrelacionados.

    H0: os dois metodos tem acuracia igual.
    H1 (unilateral): o modelo 1 tem erro MENOR que o modelo 2.

    Retorna (estatistica_dm, valor_p_unilateral).
    """
    e1 = np.asarray(erro1, dtype=float)
    e2 = np.asarray(erro2, dtype=float)
    if loss == "abs":
        l1, l2 = np.abs(e1), np.abs(e2)
    else:
        l1, l2 = e1 ** 2, e2 ** 2
    d = l1 - l2
    T = len(d)
    d_bar = d.mean()

    # variancia HAC (Newey-West), defasagem = h-1
    gamma0 = np.var(d, ddof=0)
    var_d = gamma0
    for lag in range(1, h):
        cov = np.cov(d[lag:], d[:-lag], ddof=0)[0, 1] if T > lag else 0.0
        var_d += 2 * (1 - lag / h) * cov
    var_d = max(var_d, 1e-12)

    dm_stat = d_bar / np.sqrt(var_d / T)

    # correcao HLN para amostra pequena
    correcao = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    dm_stat_corrigida = dm_stat * correcao

    p_valor = stats.t.cdf(dm_stat_corrigida, df=T - 1)  # unilateral: modelo1 melhor
    return dm_stat_corrigida, p_valor


def tabela_dm_theta_vs_todos(resultados: list[ResultadoFold], h: int) -> pd.DataFrame:
    """Compara CADA variante da familia Theta contra CADA metodo tradicional
    (matriz completa), usando o teste de Diebold-Mariano unilateral (H1: a
    variante Theta tem erro menor)."""
    # organiza erros pareados por (origem, passo_h) para cada modelo
    def erros_empilhados(modelo):
        return np.concatenate([
            r.y_real - r.y_previsto for r in resultados if r.modelo == modelo
        ])

    linhas = []
    for modelo_theta in MODELOS_THETA:
        erros_theta = erros_empilhados(modelo_theta)
        for modelo_trad in MODELOS_TRADICIONAIS:
            erros_m = erros_empilhados(modelo_trad)
            dm_stat, p_valor = teste_diebold_mariano(erros_theta, erros_m, h=h, loss="abs")
            linhas.append({
                "comparacao": f"{modelo_theta} vs {modelo_trad}",
                "DM_stat": dm_stat,
                "p_valor (Theta melhor)": p_valor,
                "Theta significativamente melhor (5%)": p_valor < 0.05,
            })
    return pd.DataFrame(linhas)




In [7]:
# ======================================================================
# 6) GRAFICOS
# ======================================================================

def _estilo_eixos(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(COR_EIXO)
    ax.spines["bottom"].set_color(COR_EIXO)
    ax.tick_params(colors=COR_EIXO)
    ax.grid(axis="y", color=COR_GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)


def grafico_serie_historica(serie: pd.Series, origens_teste: list[pd.Timestamp], caminho: Path):
    fig, ax = plt.subplots(figsize=(11, 4.5), dpi=150)
    ax.plot(serie.index, serie.values, color=COR_REAL, linewidth=1.4, label="Nivel observado")
    if origens_teste:
        ax.axvspan(origens_teste[0], serie.index[-1], color=CORES_MODELOS["Theta"], alpha=0.06,
                    label="Janela usada na validacao")
    ax.set_title("Nivel do rio — serie mensal historica", color="#0b0b0b", fontsize=12, loc="left")
    ax.set_ylabel("Nivel (cm)")
    _estilo_eixos(ax)
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(caminho, facecolor="white")
    plt.close(fig)


def grafico_previsoes_ultima_origem(resultados: list[ResultadoFold], serie: pd.Series, caminho: Path):
    """Dois paineis lado a lado (mesmo eixo Y) para nao poluir o grafico com
    7 series numa unica linha: esquerda = familia Theta, direita = metodos
    tradicionais. Assim da pra comparar visualmente cada grupo contra o
    observado sem espaguete de cores."""
    ultima_origem = max(r.origem for r in resultados)
    fold_por_modelo = {r.modelo: r for r in resultados if r.origem == ultima_origem}

    janela_hist = serie.loc[: ultima_origem].iloc[-24:]
    r_qualquer = next(iter(fold_por_modelo.values()))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150, sharey=True)
    grupos = [("Familia Theta", MODELOS_THETA), ("Metodos tradicionais", MODELOS_TRADICIONAIS)]

    for ax, (titulo, modelos) in zip(axes, grupos):
        ax.plot(janela_hist.index, janela_hist.values, color=COR_REAL, linewidth=1.6,
                label="Observado (treino)")
        ax.plot(r_qualquer.data_alvo, r_qualquer.y_real, color=COR_REAL, linewidth=2.2,
                linestyle="--", marker="o", markersize=4, label="Observado (teste)")
        for nome_modelo in modelos:
            r = fold_por_modelo[nome_modelo]
            ax.plot(r.data_alvo, r.y_previsto, color=CORES_MODELOS[nome_modelo], linewidth=1.8,
                    marker="o", markersize=3.5, label=nome_modelo)
        ax.axvline(ultima_origem, color=COR_EIXO, linewidth=1, linestyle=":")
        ax.set_title(titulo, fontsize=11, loc="left")
        _estilo_eixos(ax)
        ax.legend(frameon=False, loc="upper left", fontsize=8.5)
        ax.tick_params(axis="x", rotation=20)

    axes[0].set_ylabel("Nivel (cm)")
    fig.suptitle(f"Previsoes vs. observado — ultima origem ({ultima_origem.date()})",
                 fontsize=12, x=0.02, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(caminho, facecolor="white")
    plt.close(fig)


def grafico_barras_metricas(resumo: pd.DataFrame, caminho: Path):
    metricas = ["MAE", "RMSE", "MASE"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), dpi=150)
    ordem_modelos = resumo.index.tolist()
    cores = [CORES_MODELOS[m] for m in ordem_modelos]

    for ax, metrica in zip(axes, metricas):
        valores = resumo[metrica].values
        barras = ax.bar(ordem_modelos, valores, color=cores, width=0.6, zorder=3)
        for barra, v in zip(barras, valores):
            ax.text(barra.get_x() + barra.get_width() / 2, v, f"{v:.1f}",
                    ha="center", va="bottom", fontsize=8.5, color="#0b0b0b")
        ax.set_title(metrica, fontsize=11, loc="left")
        ax.tick_params(axis="x", rotation=35, labelsize=8)
        _estilo_eixos(ax)

    fig.suptitle("Erro medio por metodo — validacao rolling-origin (menor = melhor)",
                 fontsize=12, x=0.02, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(caminho, facecolor="white")
    plt.close(fig)


def grafico_erro_por_horizonte(tabela_h: pd.DataFrame, caminho: Path):
    """Dois paineis (mesmo eixo Y): familia Theta vs. tradicionais, para
    manter cada linha legivel em vez de 7 series sobrepostas."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=150, sharey=True)
    grupos = [("Familia Theta", MODELOS_THETA), ("Metodos tradicionais", MODELOS_TRADICIONAIS)]

    for ax, (titulo, modelos) in zip(axes, grupos):
        for nome_modelo in modelos:
            sub = tabela_h[tabela_h["modelo"] == nome_modelo].sort_values("h")
            ax.plot(sub["h"], sub["erro_abs"], color=CORES_MODELOS[nome_modelo], linewidth=1.8,
                    marker="o", markersize=4, label=nome_modelo)
        ax.set_xlabel("Passos a frente (meses)")
        ax.set_title(titulo, fontsize=11, loc="left")
        _estilo_eixos(ax)
        ax.legend(frameon=False, fontsize=8.5)

    axes[0].set_ylabel("Erro absoluto medio (cm)")
    fig.suptitle("Erro medio por horizonte de previsao", fontsize=12, x=0.02, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    fig.savefig(caminho, facecolor="white")
    plt.close(fig)


def grafico_boxplot_erros(df_metricas: pd.DataFrame, caminho: Path):
    fig, ax = plt.subplots(figsize=(10.5, 4.6), dpi=150)
    ordem_modelos = list(CORES_MODELOS.keys())
    dados = [df_metricas[df_metricas["modelo"] == m]["MAE"].values for m in ordem_modelos]
    bp = ax.boxplot(dados, tick_labels=ordem_modelos, patch_artist=True, widths=0.5)
    for patch, modelo in zip(bp["boxes"], ordem_modelos):
        patch.set_facecolor(CORES_MODELOS[modelo])
        patch.set_alpha(0.55)
        patch.set_edgecolor(CORES_MODELOS[modelo])
    for mediana in bp["medians"]:
        mediana.set_color("#0b0b0b")
    ax.set_ylabel("MAE por origem (cm)")
    ax.set_title("Distribuicao do MAE entre as origens de validacao", fontsize=12, loc="left")
    ax.tick_params(axis="x", rotation=25, labelsize=8.5)
    _estilo_eixos(ax)
    fig.tight_layout()
    fig.savefig(caminho, facecolor="white")
    plt.close(fig)




In [8]:
# ======================================================================
# MAIN
# ======================================================================

def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("== 1) Lendo e parseando o CSV bruto ==")
    serie_diaria = carregar_serie_diaria_ana(CSV_PATH, ESTACAO_CODIGO)
    print(f"Serie diaria: {serie_diaria.index.min().date()} a {serie_diaria.index.max().date()} "
          f"({len(serie_diaria)} observacoes)")

    print("\n== 2) Preparando a serie na frequencia de trabalho ==")
    serie = preparar_serie(serie_diaria, FREQ)
    print(f"Serie {FREQ}: {len(serie)} observacoes, de {serie.index.min().date()} a {serie.index.max().date()}")

    print("\n== 3) Validacao rolling-origin ==")
    resultados = validar_rolling_origin(
        serie, h=HORIZONTE, n_origens=N_ORIGENS, passo=PASSO_ENTRE_ORIGENS,
        min_treino=MIN_TREINO, periodo=PERIODO_SAZONAL,
    )

    print("\n== 4) Calculando metricas ==")
    df_metricas, resumo = montar_tabela_metricas(resultados, serie, PERIODO_SAZONAL)
    tabela_h = montar_tabela_por_horizonte(resultados)

    print("\nResumo (medias entre todas as origens, ordenado por MASE):")
    print(resumo.round(2).to_string())

    print("\n== 5) Teste de Diebold-Mariano (familia Theta vs. metodos tradicionais) ==")
    tabela_dm = tabela_dm_theta_vs_todos(resultados, h=HORIZONTE)
    print(tabela_dm.round(4).to_string(index=False))

    print("\n== 6) Salvando tabelas ==")
    df_metricas.to_csv(OUTPUT_DIR / "metricas_por_origem.csv", index=False)
    resumo.to_csv(OUTPUT_DIR / "resumo_metricas.csv")
    tabela_h.to_csv(OUTPUT_DIR / "erro_por_horizonte.csv", index=False)
    tabela_dm.to_csv(OUTPUT_DIR / "teste_diebold_mariano.csv", index=False)

    print("\n== 7) Gerando graficos ==")
    origens_teste = sorted({r.origem for r in resultados})
    grafico_serie_historica(serie, origens_teste, OUTPUT_DIR / "01_serie_historica.png")
    grafico_previsoes_ultima_origem(resultados, serie, OUTPUT_DIR / "02_previsoes_vs_observado.png")
    grafico_barras_metricas(resumo, OUTPUT_DIR / "03_metricas_por_metodo.png")
    grafico_erro_por_horizonte(tabela_h, OUTPUT_DIR / "04_erro_por_horizonte.png")
    grafico_boxplot_erros(df_metricas, OUTPUT_DIR / "05_boxplot_mae.png")

    print(f"\nConcluido. Resultados em: {OUTPUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

== 1) Lendo e parseando o CSV bruto ==
Serie diaria: 1982-07-05 a 2025-02-28 (15149 observacoes)

== 2) Preparando a serie na frequencia de trabalho ==
[preparar_serie] lacunas antes=6, depois da interpolacao=0
Serie MS: 512 observacoes, de 1982-07-01 a 2025-02-01

== 3) Validacao rolling-origin ==
[rolling-origin] 6 origens, treino inicial=470 obs, treino final=500 obs, horizonte=12
[SARIMA] ordem escolhida por AIC: ((2, 1, 1), (0, 1, 1, 12)) (AIC=5500.2)
  origem 2021-08-01 (treino=470 obs) — OK
  origem 2022-02-01 (treino=476 obs) — OK
  origem 2022-08-01 (treino=482 obs) — OK
  origem 2023-02-01 (treino=488 obs) — OK
  origem 2023-08-01 (treino=494 obs) — OK
  origem 2024-02-01 (treino=500 obs) — OK

== 4) Calculando metricas ==

Resumo (medias entre todas as origens, ordenado por MASE):
                       MAE    RMSE  MASE
modelo                                  
Naive Sazonal       155.07  196.82  0.93
SARIMA              161.53  201.54  0.97
ETS (Holt-Winters)  176.77  202.4